In [2]:
# ==========================================================
# 01_preprocessing.py
# Preprocess CASME II Face Landmarks for SSGNN
# ==========================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import joblib
import os

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------

CSV_PATH = "D:\\IDE Repo\\Dl-net\\data\\casme2-preprocessed-v2\\train.csv"      # change if necessary
SAVE_DIR = "processed"

os.makedirs(SAVE_DIR, exist_ok=True)

# ----------------------------------------------------------
# Read CSV
# ----------------------------------------------------------

df = pd.read_csv(CSV_PATH)

print("=" * 60)
print("Original Dataset")
print(df.shape)
print("=" * 60)

# ----------------------------------------------------------
# Keep only detected landmarks
# ----------------------------------------------------------

if "landmarks_detected" in df.columns:
    df = df[df["landmarks_detected"] == True]

print("After removing failed detections :", df.shape)

# ----------------------------------------------------------
# Find Label Column
# ----------------------------------------------------------

possible_labels = [
    "emotion",
    "Emotion",
    "class",
    "Class",
    "label",
    "Label"
]

label_column = None

for col in possible_labels:
    if col in df.columns:
        label_column = col
        break

if label_column is None:
    raise Exception(
        "Cannot find emotion/class column."
    )

print("Label Column :", label_column)

# ----------------------------------------------------------
# Landmark Columns
# ----------------------------------------------------------

x_cols = sorted(
    [c for c in df.columns if c.startswith("x_")],
    key=lambda x: int(x.split("_")[1])
)

y_cols = sorted(
    [c for c in df.columns if c.startswith("y_")],
    key=lambda x: int(x.split("_")[1])
)

z_cols = sorted(
    [c for c in df.columns if c.startswith("z_")],
    key=lambda x: int(x.split("_")[1])
)

print()

print("Number of X landmarks :", len(x_cols))
print("Number of Y landmarks :", len(y_cols))
print("Number of Z landmarks :", len(z_cols))

assert len(x_cols) == len(y_cols)

num_landmarks = len(x_cols)

print("Detected Landmarks :", num_landmarks)

# ----------------------------------------------------------
# Create Feature Matrix
# ----------------------------------------------------------

features = []

for i in range(num_landmarks):

    features.append(x_cols[i])
    features.append(y_cols[i])

X = df[features]

print()

print("Feature Matrix :", X.shape)

# ----------------------------------------------------------
# Missing Value Imputation
# ----------------------------------------------------------

imputer = SimpleImputer(strategy="mean")

X = imputer.fit_transform(X)

joblib.dump(
    imputer,
    os.path.join(SAVE_DIR, "imputer.pkl")
)

# ----------------------------------------------------------
# Encode Labels
# ----------------------------------------------------------

encoder = LabelEncoder()

y = encoder.fit_transform(df[label_column])

joblib.dump(
    encoder,
    os.path.join(SAVE_DIR, "label_encoder.pkl")
)

print()

print("Classes")

for i, c in enumerate(encoder.classes_):
    print(i, "->", c)

# ----------------------------------------------------------
# Train/Test Split
# ----------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

print()

print("Train :", X_train.shape)
print("Test  :", X_test.shape)

# ----------------------------------------------------------
# Save
# ----------------------------------------------------------

np.save(
    os.path.join(SAVE_DIR, "X_train.npy"),
    X_train
)

np.save(
    os.path.join(SAVE_DIR, "X_test.npy"),
    X_test
)

np.save(
    os.path.join(SAVE_DIR, "y_train.npy"),
    y_train
)

np.save(
    os.path.join(SAVE_DIR, "y_test.npy"),
    y_test
)

print()

print("=" * 60)
print("Preprocessing Completed")
print("=" * 60)

Original Dataset
(13615, 14)
After removing failed detections : (13615, 14)
Label Column : class

Number of X landmarks : 0
Number of Y landmarks : 0
Number of Z landmarks : 0
Detected Landmarks : 0

Feature Matrix : (13615, 0)


ValueError: at least one array or dtype is required